# Synth-1 — обучение на TPU

**Как пользоваться: Save & Run All. Больше ничего.**

Ноутбук сам разбирается, что делать. Нет снимка на Hub — начинает с нуля, сняв
по дороге замер скорости. Снимок есть — молча продолжает с него. Каждая сессия
берёт свои восемь часов и уходит, оставив чекпоинт; следующая подхватывает.

Перед первым запуском: в правой панели включить **TPU VM v5e-8** и добавить
секрет `HF_TOKEN` (Add-ons → Secrets). Без секрета не будет ни корпуса, ни
чекпоинтов. И подключить входом output прошлой версии (**Add Input → Your
Work → Notebook Output**) — не только после аварии, а каждый раз: на этом
держится освобождение квоты Hub, см. 6б.

> **Ноутбук на Kaggle — копия, а не сам файл.** Он живёт в репозитории
> (`notebooks/kaggle_tpu.ipynb`, собирается скриптом) и попадает на Kaggle
> импортом. Правки в репозитории сами туда не доезжают: сессия 07.09 запустила
> версию без `--free-first`, хотя в коде он был уже неделю. Перед запуском
> стоит убедиться, что на Kaggle лежит свежая версия.

> **Не импортируй jax в ячейках.** TPU достаётся ровно одному процессу. Если
> ядро ноутбука тронет устройства, оно захватит чип, и обучение упадёт с
> `open(/dev/vfio/0): Device or resource busy`. Всё, что касается TPU, здесь
> запускается отдельным процессом — включая проверку числа чипов.

## 1. Хозяйство сессии

In [ ]:
import subprocess, os, sys, time, re

# SYNTH_MODE = "chat" в ячейке-загрузчике — поговорить с моделью (demo/chat.py).
# Чату TPU не обязателен: годится и GPU T4 ×2 (очереди почти нет), и CPU
# (медленно). Проверки «8 чипов TPU» для него не делаются.
CHAT = globals().get("SYNTH_MODE") == "chat"

# Отпечаток этого файла. Сверяется со склонированной копией ниже: ячейка
# Kaggle тянет сценарий через API GitHub, и он может приехать устаревшим.
BUILT = "?"

T0 = time.time()
# Лимит сессии — девять часов. Полчаса резерва: на установку пакетов, на
# выгрузку последнего снимка и на то, чтобы Kaggle не убил нас на полуслове.
SESSION_HOURS = 8.5
WORK = "/kaggle/working/synth-1"

def sh(*cmd):
    return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()

def free_gb(path="/kaggle/working"):
    s = os.statvfs(path)
    return s.f_bavail * s.f_frsize / 1024**3

def left_hours():
    return SESSION_HOURS - (time.time() - T0) / 3600

SECRETS = []          # значения, которых не должно быть в логе

def mask(text):
    """Убрать секреты из строки перед печатью.

    Токен уезжает в команду `git clone`, а лог ноутбука сохраняется вместе с
    версией и виден всем, кому её показывают. Один раз напечатать достаточно,
    чтобы токен пришлось отзывать.
    """
    for secret in SECRETS:
        if secret:
            text = text.replace(secret, "***")
    return text

def run(*cmd, check=True):
    """Запуск с выводом по мере появления.

    Именно по мере появления, а не в конце. Прогон идёт восемь часов, и если
    собирать его вывод в память, то при обрыве сессии в логе не останется
    ничего — ни лосса, ни причины. Возвращаем код и хвост, по хвосту
    разбираемся, что случилось.
    """
    print("$", mask(" ".join(cmd)), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=env)
    tail = []
    for line in p.stdout:
        line = mask(line)
        print(line, end="", flush=True)
        tail.append(line)
        if len(tail) > 500:
            tail.pop(0)
    p.wait()
    print(f"[код {p.returncode}, свободно {free_gb():.1f} ГБ, "
          f"осталось {left_hours():.1f} ч]", flush=True)
    if p.returncode and check:
        raise SystemExit(mask(f"упало: {' '.join(cmd)}"))
    return p.returncode, "".join(tail)

print(f"сценарий сессии: {BUILT}")
print(sh("nproc"), "ядер CPU")
print(sh("free", "-g"))
print(sh("df", "-h", "/kaggle/working"))
# TPU-машина Kaggle — 96 ядер. 4 ядра — CPU-образ (gcr.io/kaggle-images/python):
# так приходили все запуски по расписанию и ручной 21.10. Ставить JAX и ждать
# чипов там незачем.
if os.cpu_count() < 32 and not CHAT:
    raise SystemExit(
        "нет TPU: машина на CPU-образе. В редакторе ноутбука: Settings → "
        "Accelerator → TPU VM v5e-8 (заново выбрать, даже если уже стоит), "
        "отключить Schedule, затем Save Version → Save & Run All.")

## 2. Окружение

JAX на образе Kaggle бывает старее нужного, поэтому ставим явно. Всё через
`subprocess`: под Save & Run ячейки с `!` ведут себя иначе, чем в интерактивном
запуске, и на этом в прототипе на MiniCPM уже спотыкались.

In [ ]:
import shutil as _sh
if CHAT and os.cpu_count() < 32:
    # Не TPU-машина: JAX под GPU, если он есть, иначе под CPU.
    run(sys.executable, "-m", "pip", "install", "-q", "-U",
        "jax[cuda12]" if _sh.which("nvidia-smi") else "jax")
else:
    run(sys.executable, "-m", "pip", "install", "-q", "-U",
        "jax[tpu]", "-f", "https://storage.googleapis.com/jax-releases/libtpu_releases.html")
run(sys.executable, "-m", "pip", "install", "-q",
    "tokenizers", "safetensors", "huggingface_hub", "pyyaml")

## 3. Код проекта

Репозиторий приватный, поэтому нужен секрет `GITHUB_TOKEN` (Add-ons → Secrets):
токен с правом чтения `ssane-labs/synth-1`. Раньше доступ давала связка
Kaggle↔GitHub у той версии ноутбука, но она не переносится при импорте — сессия
31.08.2026 из-за этого не дожила до первого шага.

Токен в лог не попадает: команды печатаются через `mask()`, а в `.git/config`
кладётся адрес без него — иначе токен уехал бы в output версии вместе с
репозиторием.

In [ ]:
GITHUB = "https://github.com/ssane-labs/synth-1.git"
# Ветка кода. Обычно main; ячейка-загрузчик может задать SYNTH_BRANCH до exec,
# чтобы сессия пошла кодом из ветки, ещё не влитой в main (так шла 04.10.2026).
BRANCH = globals().get("SYNTH_BRANCH", "main")
print(f"ветка: {BRANCH}")

GH_TOKEN = os.environ.get("GITHUB_TOKEN", "")
if not GH_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        GH_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as exc:
        print(f"секрета GITHUB_TOKEN нет ({type(exc).__name__}) — "
              f"пробуем без него, вдруг репозиторий открыли")
SECRETS.append(GH_TOKEN)

# Адрес с токеном используется только в аргументах команды и никогда не
# сохраняется: remote остаётся чистым, fetch получает адрес явно.
AUTH = (GITHUB.replace("https://", f"https://x-access-token:{GH_TOKEN}@")
        if GH_TOKEN else GITHUB)

if not os.path.exists(WORK):
    code_, _ = run("git", "clone", "--depth", "1", "--branch", BRANCH, AUTH, WORK,
                   check=False)
    if code_:
        raise SystemExit(chr(10).join([
            "Не удалось склонировать репозиторий. Он приватный, значит "
            "нужен секрет GITHUB_TOKEN:",
            "  1. github.com -> Settings -> Developer settings -> "
            "Personal access tokens -> Fine-grained",
            "  2. доступ к ssane-labs/synth-1, право Contents: Read",
            "  3. Kaggle -> Add-ons -> Secrets -> добавить как GITHUB_TOKEN",
            "Без кода проекта сессия всё равно ничего не посчитает.",
        ]))
    run("git", "-C", WORK, "remote", "set-url", "origin", GITHUB)
else:
    run("git", "-C", WORK, "fetch", "--depth", "1", AUTH, BRANCH)
    run("git", "-C", WORK, "reset", "--hard", "FETCH_HEAD")
os.chdir(WORK)
run("git", "-C", WORK, "log", "--oneline", "-3")

# Сверка сценария с репозиторием. Ячейка Kaggle тянет этот файл через API
# GitHub, и 15.09.2026 он приехал устаревшим: репозиторий склонировался свежий,
# обучение пошло новым кодом, а сценарий сессии остался прежним — перебор
# померил снятого подозреваемого, разрез по частям не запустился вовсе, и в
# логе об этом не было ни слова. Полчаса TPU и весь смысл замера.
_fresh = re.search(r'^BUILT = "(\w+)"',
                   open(f"{WORK}/notebooks/kaggle_run.py", encoding="utf-8").read(),
                   re.M)
if _fresh and _fresh.group(1) != BUILT:
    print(chr(10).join([
        "",
        "!" * 70,
        f"СЦЕНАРИЙ СЕССИИ УСТАРЕЛ: выполняется {BUILT}, в репозитории "
        f"{_fresh.group(1)}.",
        "Обучение пойдёт свежим кодом (он из клона), а вот порядок сессии —",
        "замеры, их параметры, всё в этой ячейке — старый.",
        "Чинится перезапуском: API GitHub отдаёт свежую копию через несколько",
        "минут. Если повторяется — в ячейке ноутбука лежит не однострочный",
        "загрузчик, а вставленная руками копия скрипта; заменить на строку из",
        "шапки notebooks/kaggle_run.py.",
        "!" * 70,
        "",
    ]), flush=True)
else:
    print(f"сценарий сессии совпадает с репозиторием ({BUILT})")

## 4. Чипы на месте?

Должно быть восемь. Если меньше — TPU не выдан, и дальше идти незачем: прогон на
одном чипе съест квоту и ничего не покажет.

Проверка идёт отдельным процессом и **сразу его закрывает**, освобождая чип.

In [ ]:
run(sys.executable, "-c",
    "import jax; d = jax.devices(); print(len(d), 'устройств:', d[0].device_kind); "
    + ("" if CHAT else "assert len(d) == 8, f'ожидалось 8 чипов, видно {len(d)}'"))

## 5. Первый это запуск или продолжение?

Вопрос решается одним обращением к Hub: есть ли там снимок прогона `pretrain`.
Есть — значит сессия продолжает начатое, и всё, что ниже до самого обучения,
пропускается само. Нет — значит это старт, и перед восемью часами счёта имеет
смысл потратить пять минут на замер скорости.

Ядро здесь трогает только сеть, не TPU.

In [ ]:
from huggingface_hub import HfApi
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception as exc:
    print(f"секрет HF_TOKEN не достался из Kaggle ({exc}) — надеемся на окружение")

REPO = "sekund0chka/synth-1-pretrain"
try:
    files = HfApi(token=os.environ.get("HF_TOKEN")).list_repo_files(REPO)
    steps = sorted(int(f.split("/")[0][5:]) for f in files
                   if f.startswith("step-") and f.endswith("meta.json"))
except Exception as exc:
    print(f"Hub не ответил или репозитория ещё нет ({type(exc).__name__})")
    steps = []

FIRST = not steps
print(f"снимков на Hub: {len(steps)}" + (f", последний шаг {steps[-1]:,}" if steps else ""))
print("это старт с нуля" if FIRST else "продолжаем начатое")

## 6. Куда уходит время шага

Только на первом запуске. Замер стоит пять минут и отвечает на вопрос, который
иначе стоит недель: сколько процентов чипа мы используем и какой механизм
съедает шаг. Разложение печатается в лог — читать потом, спокойно.

In [ ]:
# SWEEP: снять разложение времени шага — куда уходит 94% чипа.
#
#   False                не снимать, вся сессия уходит обучению
#   "Ньютон,Muon"        про ортогонализацию, ~полчаса
#   True                 все пятнадцать вариантов, ~полтора часа
#
# Каждый вариант компилируется заново, и компиляция — это почти всё время
# замера, поэтому цена растёт по числу вариантов, а не по числу шагов.
# Что бы здесь ни стояло, обучение получит остаток часов и ничего не потеряет:
# замер идёт до него и Hub не трогает.
#
# MODE: что вообще делает сессия.
#
#   "pretrain"    обычная сессия: замеры, квота Hub, продолжение обучения
#   "ablations"   батарея абляций на конфигурации 88M, в конце таблица
#   "anneal"      отжиг LR на НАСТОЯЩЕЙ модели из снимка, без записи снимков
#   "final"       настоящий отжиг (этап C): под своим прогоном 'anneal', со
#                 снимками, стартует со свежего снимка плато и его не трогает
#   "context"     растяжка контекста 2048 → 8192 (curriculum этапа B и
#                 начало C2): прогон 'context' со снимка 'anneal', смесь
#                 'anneal', разогрев LR до CONTEXT_LR от пика и спад до пола
#
# РЕШЕНИЕ 04.10.2026: претрейн закончен. Настоящий отжиг 26.09 прошёл
# 141,888 → 161,888 за 5.41 ч и дал отложенный лосс 2.1474 → 1.8300
# (ppl 6.2) против 1.9170/1.9195 у коротких отжигов; к концу спуск вышел на
# полку (−0.004 на 1000 шагов). Вся модель при этом училась только на длине
# 2048, а curriculum 2k → 4k → 8k из PLAN.md не пройден — это и есть
# следующий этап. Растяжка идёт сразу на 8192: окно 16 слоёв из 20 — 1024,
# длину видят только четыре глобальных слоя, и промежуточная ступень 4096
# им мало что даёт.
#
# РЕШЕНИЕ 24.09.2026: плато больше не копит. Два коротких отжига по 2000
# шагов дали 1.9170 со снимка 84,425 (5.53B токенов) и 1.9195 со снимка
# 113,780 (7.46B) — то есть 29 тысяч шагов на пиковом LR не купили ничего,
# а следующие 28 тысяч (до 141,888, 9.30B) судя по отложенному лоссу тоже.
# Поэтому плато заканчиваем и отжигаем по-настоящему, длинно: FINAL_ANNEAL_STEPS
# шагов со снимка 141,888 под прогоном 'anneal' (репозиторий
# synth-1-anneal). Снимки плато в synth-1-pretrain остаются как есть, и к
# плато можно вернуться.
#
# ГЛАВНОЕ, ЧТО ВЫЯСНИЛОСЬ 22.09.2026: плато трёх сессий было работой
# расписания, а не болезнью модели.
#
# Отжиг на 2000 шагов со снимка 84,425 дал 2.0854 → 1.9170 (ppl 8.05 → 6.80),
# наклон −0.0929 на 1000 шагов при t = −19.9. Для сравнения, те же замеры на
# плато 55,030…84,425 давали наклон +0.00002 при t = +0.04, то есть ровно
# ноль. Модель здорова, переучивать нечего, менять в конструкции нечего.
#
# Отсюда правило, которое стоило трёх сессий поисков:
#
#     ПРОГРЕСС НА ПЛАТО WSD ОТЛОЖЕННЫМ ЛОССОМ НЕ МЕРЯЕТСЯ.
#     Он меряется коротким отжигом с той же точки.
#
# LR держится на пике до 88% длины (отжиг с 176,000 шага), и всё это время
# отложенный лосс стоит по построению. Чтобы увидеть настоящий прогресс,
# раз в несколько сессий ставим MODE = "anneal": те же 2000 шагов с текущего
# снимка, снимки не пишутся, точка продолжения не сдвигается. Числа сравнимы
# между собой только при одинаковой длине отжига.
#
#     эталон: снимок 84,425 (5.53B токенов) → 1.9170
#
MODE = "chat" if CHAT else "auto"
# MODE = "chat" — поговорить с последней моделью самому (demo/chat.py): в
# редакторе ноутбука Run All, в выводе — ссылка на веб-чат.

# АВТОПИЛОТ (MODE = "auto"). Ноутбук запускается по расписанию Kaggle и сам
# решает, что делать: идёт по плану STAGES, пропуская этапы, у которых на
# Hub уже лежит законченный снимок, и проходит этапы подряд, пока у сессии
# хватает часов. Этап закончен, когда его последний снимок стоит на шаге
# anneal_from + steps (anneal_from записан в снимке).
#
# Каждый этап — новый прогон со своим репозиторием на Hub, старт со снимка
# предыдущего, разогрев LR до lr × пиковый и спад до пола. Снимки прошлых
# этапов не трогаются.
#
# В конце лога — строка «АВТОПИЛОТ: …»: что сделано и нужна ли работа в коде.
# По ней смотрит и человек, и сессия Claude, которая следит за расписанием.
STAGES = [
    # Растяжка контекста 2048 → 8192, сделана 06.10.2026: отложенный лосс
    # 1.8300 → 1.7771 на 2048 и 2.0750 → 1.6778 на 8192.
    {"run": "context", "init": "anneal", "steps": 24_000, "mixture": "anneal",
     "seq": 8192, "lr": 0.25, "warmup": 500},
    # D1: ответ по документам со ссылками и честный отказ (data/sft.py).
    # ~240K вопросов, ~3,800 шагов на эпоху; две эпохи. LR ×0.1: этап учит
    # формат и поведение, а не знания, и не должен сносить претрейн.
    {"run": "d1", "init": "context", "steps": 7_000, "mixture": "d1",
     "seq": 8192, "lr": 0.1, "warmup": 200, "eval": "d1"},
    # D1 сделан 19.10: атрибуция 97%, честный отказ 76%, ложный отказ 8%
    # (SQuAD — 28%). Отложенный лосс встал с 188,000: вторая эпоха лишняя.
    #
    # D2: поиск (data/search_sft.py). Траектории OpenResearcher, сжатые под
    # 8192, ~3 тыс. вопросов; вперемешку с D1 (SQuAD вдвое) и претрейном,
    # 50/30/20 по строкам. ~375 шагов на эпоху поиска, 2,000 шагов — ~5 эпох
    # поиска при ~15 минутах TPU. Первый запуск качает 7.4 ГБ траекторий и
    # перекладывает их на всех ядрах — минут десять.
    {"run": "d2", "retired": True, "init": "d1", "steps": 2_000, "mixture": "d2",
     "seq": 8192, "lr": 0.1, "warmup": 100, "eval": "d2"},
    # D2 сделан 21.10: начинает с поиска в 63%+ вопросов, D1-навыки удержаны
    # (честный отказ 77%, ложный 9%, атрибуция 95%). Факты по памяти поплыли.
    #
    # E: калибровка уверенности (data/calib.py). Сначала снимок d2 сам отвечает
    # на 4,500 вопросов D1 (по 2 ответа) и 8,000 NQ-open без документов (по 4),
    # ответы сверяются с эталоном — минут 20–30 на восьми чипах; данные
    # уходят на Hub (synth-1-e-data) и при обрыве не генерируются заново.
    # Потом 600 шагов: головка учится предсказывать свою ошибку, язык — только
    # на верных ответах, на незнакомых NQ-вопросах — начинать с <|search|>.
    {"run": "e", "init": "d2", "steps": 600, "mixture": "e", "retired": True,
     "seq": 8192, "lr": 0.1, "warmup": 50, "eval": "e",
     "prepare": ["data/calib.py", "build", "--init", "d2"]},
    # Очередь на TPU — больше суток, а этап — минуты, поэтому один запуск
    # проходит всю цепочку E → F → G. Данные F и G — те же ответы, что
    # собраны для E (prepare их только скачивает, если сессия новая).
    #
    # F: предпочтения, ORPO (data/prefs.py): отказ > выдумка, верно > неверно,
    # поиск > выдумка. LR вдвое ниже: этап шлифует, а не учит новому.
    {"run": "f", "init": "e", "steps": 400, "mixture": "f", "retired": True,
     "seq": 8192, "lr": 0.05, "warmup": 30, "eval": "f",
     "prepare": ["data/calib.py", "build", "--init", "d2"]},
    # G: личность (data/persona.py) — 3% строк, остальное — всё прошлое.
    {"run": "g", "init": "f", "steps": 150, "mixture": "g", "retired": True,
     "seq": 8192, "lr": 0.05, "warmup": 20, "eval": "g",
     "prepare": ["data/calib.py", "build", "--init", "d2"]},
    # Цепочка 23.10 прошла за 2 часа, итоги — STATUS.md. Вторая версия,
    # опять одним запуском, с исправлениями:
    #   d2v2 — D2 без болтовни учителя о среде (search_sft.tidy): одна мысль
    #          перед действием; 1,500 шагов — D2 вставал к ~1,100;
    #   ev2  — данные E с валидации D1 (модель её не видела), мягкий судья,
    #          лосс на всём поисковом действии;
    #   fv2  — пары из новых данных;
    #   gv2  — личность 15% строк, LR ×0.1, 300 шагов.
    # Замеры сравнивают с первой версией (baseline), поисковик — d2 с gv2
    # (снимки e, f, g первой версии можно удалить с Hub ради места).
    {"run": "d2v2", "retired": True, "init": "d1", "steps": 1_500, "mixture": "d2",
     "seq": 8192, "lr": 0.1, "warmup": 100, "eval": "d2", "baseline": "d2"},
    {"run": "ev2", "retired": True, "init": "d2v2", "steps": 600, "mixture": "e",
     "seq": 8192, "lr": 0.1, "warmup": 50, "eval": "e", "baseline": "e",
     "prepare": ["data/calib.py", "build", "--init", "d2v2"]},
    {"run": "fv2", "retired": True, "init": "ev2", "steps": 400, "mixture": "f",
     "seq": 8192, "lr": 0.05, "warmup": 30, "eval": "f", "baseline": "f",
     "prepare": ["data/calib.py", "build", "--init", "d2v2"]},
    # 29.10: после удаления e/f/g с Hub автопилот счёл их недоделанными и
    # переучил (1.3 ч). Теперь выведенные этапы помечены retired — их не
    # трогают, есть они на Hub или нет. Поиск d2 со средой уже измерен
    # (STATUS.md, 03.11) — гоняем только финальный снимок, это −80 минут.
    # Личность: g с 15% строк и LR ×0.05 начал отвечать «I'm …», но имя не
    # выучил — gv2 400 шагов, LR ×0.1, доля 25% (data/persona.py).
    {"run": "gv2", "retired": True, "init": "fv2", "steps": 400, "mixture": "g",
     "seq": 8192, "lr": 0.1, "warmup": 30, "eval": "g", "baseline": "fv2",
     "prepare": ["data/calib.py", "build", "--init", "d2v2"]},
    # 05.11: цепочка v2 прошла, но поиск плохой — модель открывала example.com
    # и r.jina.ai вместо поиска (копировала учителя OpenResearcher), F сделал
    # её пугливой (ложный отказ на SQuAD 64%), личность была заучена фразами.
    # Третья версия, одним запуском, без учителя вовсе:
    #   d2v3 — поиск только из открытых датасетов с коммерческой лицензией
    #          (data/wiki_search.py): HotpotQA, MuSiQue (подвопросы как
    #          запросы), SQuAD; траектории собраны и проверены в среде замера;
    #   ev3  — E-данные со снимка d2v3;
    #   fv3  — плюс пары «эталон > неверно/ложный отказ»;
    #   gv3  — живой чат: OASST2 (Apache-2.0), личность из системного
    #          промпта, реакции и эмодзи (data/chat.py).
    {"run": "d2v3", "retired": True, "init": "d1", "steps": 2_000, "mixture": "d2w",
     "seq": 8192, "lr": 0.1, "warmup": 100, "eval": "d2", "baseline": "d2",
     "env": ["--only-hotpot"]},
    {"run": "ev3", "retired": True, "init": "d2v3", "steps": 600, "mixture": "e",
     "seq": 8192, "lr": 0.1, "warmup": 50, "eval": "e", "baseline": "ev2",
     "prepare": ["data/calib.py", "build", "--init", "d2v3"]},
    {"run": "fv3", "retired": True, "init": "ev3", "steps": 400, "mixture": "f",
     "seq": 8192, "lr": 0.05, "warmup": 30, "eval": "f", "baseline": "fv2",
     "prepare": ["data/calib.py", "build", "--init", "d2v3"]},
    {"run": "gv3", "retired": True, "init": "fv3", "steps": 600, "mixture": "chat",
     "seq": 8192, "lr": 0.1, "warmup": 30, "eval": "g", "baseline": "fv3",
     "prepare": ["data/calib.py", "build", "--init", "d2v3"]},
    # 11.11: d2v3 нашёл вдвое больше (HotpotQA 25% против 13%) и выдумывал
    # в 2.5 раза реже, но в 48% случаев отвечал «The sources don't say.» —
    # пользователь на этом уходит. ev3 упал на подготовке данных (KeyError:
    # у вопросов NQ нет эталона). Четвёртая версия — продукт, а не замер:
    #   ответ — сводка, как у поисковика: прямой ответ и факты со ссылками;
    #   не нашла — ищет ещё раз другими словами, потом говорит, что нашлось
    #   близкого, и предлагает искать иначе (sft.present, wiki_search);
    #   20–30% запросов — ключевыми словами, как в поисковике.
    # После gv4 — разговоры целиком (evals/demo.py): как видит пользователь.
    {"run": "d2v4", "init": "d1", "steps": 2_000, "mixture": "d2w",
     "seq": 8192, "lr": 0.1, "warmup": 100, "eval": "d2", "baseline": "d2v3",
     "env": ["--only-hotpot"]},
    {"run": "ev4", "init": "d2v4", "steps": 600, "mixture": "e",
     "seq": 8192, "lr": 0.1, "warmup": 50, "eval": "e", "baseline": "ev2",
     "prepare": ["data/calib.py", "build", "--init", "d2v4"]},
    {"run": "fv4", "init": "ev4", "steps": 400, "mixture": "f",
     "seq": 8192, "lr": 0.05, "warmup": 30, "eval": "f", "baseline": "fv2",
     "prepare": ["data/calib.py", "build", "--init", "d2v4"]},
    {"run": "gv4", "init": "fv4", "steps": 600, "mixture": "chat",
     "seq": 8192, "lr": 0.1, "warmup": 30, "eval": "g", "baseline": "fv4",
     "prepare": ["data/calib.py", "build", "--init", "d2v4"]},
]

# Этап растяжки контекста. Батч 8 × 8192 — те же 65,536 токенов на шаг, что
# 32 × 2048. LR разогревается от пола до CONTEXT_LR × пиковый за 500 шагов и
# спадает до пола к концу этапа. Множитель 0.25 — компромисс: этап короткий и
# идёт по отожжённым весам, полный пик снёс бы отжиг, а пол ничего не
# сдвинул бы в глобальных слоях. Длина этапа подобрана под замер скорости
# 04.10.2026 (shard_map, см. synth1/fsdp.py) — с запасом в одну сессию.
CONTEXT_STEPS = 24_000
CONTEXT_SEQ = 8192
CONTEXT_LR = 0.25

# Длина настоящего отжига. ~12% всей длины (20,000 из ~162,000): у WSD спад
# на 10-20% длины — обычная доля, и длинный спад даёт больше короткого. При
# ~0.95 с/шаг это ~5.3 ч, одна сессия. Не уложится — следующая сессия с тем же
# MODE продолжит по той же кривой LR: начало отжига записано в снимке.
FINAL_ANNEAL_STEPS = 20_000

# Длина отжига в режиме "anneal". Менять нельзя без нужды: числа разных
# отжигов сравнимы только при одинаковой длине. 2000 шагов — это ~40 минут.
ANNEAL_STEPS = 2000

# ANNEAL_FIRST: в режиме "pretrain" сперва снять отжиг с текущего снимка, потом
# в той же сессии продолжить обычный претрейн с того же снимка. Отжиг снимков
# не пишет, точка продолжения не сдвигается, так что претрейн ничего не теряет,
# кроме ~45 минут. Режим "anneal" сам по себе кончает сессию после отжига, и
# остальные семь часов пропадают — поэтому отдельно им больше не пользуемся.
#
# Сессия 23.09.2026 прошла 84,425 → 113,780 на пиковом LR, отложенный лосс
# стоял (2.085 → 2.123, t = +0.99), как и должен на плато. Купили ли что-то
# эти 29,355 шагов, видно только отжигом против эталона 1.9170. Заодно отжиг
# ответит, навредила ли нестабильность той сессии: шум отложенного лосса
# 0.068 против 0.014, всплески +0.24 на 105,000 и +0.12 на 110,000.
#
# После ответа вернуть False: отжиг нужен раз в несколько сессий, не каждую.
ANNEAL_FIRST = False


# Сколько часов квоты не жалко на батарею. 0 — сколько даст сессия.
# Батарея сама укладывается в бюджет: замеряет первый прогон и дальше
# запускает очередной, только если он целиком помещается в остаток.
BUDGET_HOURS = 2.5

# PROBE_LR: после батареи снять пробу на НАСТОЯЩЕЙ модели из снимка — два
# прогона по 1000 шагов с одной точки, на текущем LR и на этом множителе.
# Батарея меряет 88M на 2000 шагов от нуля, а плато стоит на 579M и 5.5B
# токенов; перенос между ними — гипотеза, и проверяется он здесь. Снимки не
# пишутся, точка продолжения не сдвигается, обучение после пробы не идёт.
# 0 — пробу не снимать.
PROBE_LR = 2.0

# Что меряем в этот раз.
#
# Подозреваемые прошлых сессий закрыты оба: внимание блоками снято 07.09
# (256 — лучший из четырёх размеров), Муон снят 14.09 (отказ от него экономит
# 9.6% шага). Прикидка «Ньютон–Шульц стоит 20 TFLOP при 32 TFLOP шага»
# оказалась неверной: 90% времени шага — сама модель.
#
# Причину нашли не в них. Шаг идёт 915 мс при пределе 59 мс по счёту и
# 73–122 мс по полосе памяти: теряется между ними — на том, что промежуточные
# тензоры ездят в память и обратно. Главный такой тензор — матрица внимания
# [C,S]: блочный путь её материализует, и она трижды за шаг проходит через
# HBM. Сплавленное ядро Pallas держит её в VMEM и наружу не отдаёт.
#
# Теперь оно включено по умолчанию (cfg.attn_impl="splash"), а этот перебор
# отвечает, сколько оно дало и что делать дальше:
#
#   внимание блочное        прежний путь — против нового, это главное число
#   без пересчёта           remat ставили ради матрицы внимания; её больше
#                           нет в памяти, и пересчёт может стать не нужен
#   эксперты со свёрткой    одна батчевая ось в einsum вместо двух
#   splash блоком 256       размер блока ядра
#
# Выключено 23.09.2026. Сплавленное ядро замерено 21.09 и откачено (медленнее
# на 16.3%), умолчание снова блочное — поэтому строки «блочное» и «блоком 256»
# перебор даже не исполняет, а остальные повторяют числа 14.09. Сессия 23.09
# отдала на это и на разрез ~25 минут и не узнала ничего нового. Включать,
# когда в модели или в пути внимания появится что-то, что стоит мерить.
SWEEP = False

# PARTS: разрез шага по частям модели (evals/parts.py). Пять-десять минут (в
# основном компиляция, по одной на часть), и это
# единственное место, где видно, во что упирается каждая часть по отдельности.
# Выключен вместе с SWEEP: числа 23.09 совпали с 14.09.
PARTS = False

# SPEED: короткий замер скорости (~6 минут с компиляцией). Та же скорость видна
# в логе обучения, в столбце «мс/шаг», так что отдельно он не нужен.
SPEED = False

# Батч тот же, что в обучении: на восьмёрке доля Ньютона-Шульца вчетверо выше
# рабочей, и разложение получается не про тот режим, в котором мы живём.
#
# Короткий замер идёт всегда: он стоит четыре минуты и печатает MFU, по
# которому видно, не изменилось ли железо под нами. Полное разложение — под
# SWEEP: оно стоит полтора часа из восьми, а сессия сейчас нужна под обучение.
if MODE != "pretrain":
    print(f"режим {MODE!r}: замеры скорости пропускаем, время нужно батарее")
    SWEEP, PARTS = False, False

if MODE == "pretrain" and SPEED:
    run(sys.executable, "evals/speed.py", "--preset", "default",
        "--seq-len", "2048", "--batch-size", "32", "--steps", "20", check=False)
if SWEEP:
    cmd = [sys.executable, "evals/speed.py", "--preset", "default",
           "--seq-len", "2048", "--batch-size", "32", "--steps", "10", "--sweep"]
    if isinstance(SWEEP, str):
        cmd += ["--only", SWEEP]
    run(*cmd, check=False)
else:
    print("разложение выключено (SWEEP=False)")

# Разрез шага по частям модели. Три минуты, один раз за сессию, и он отвечает
# на вопрос, на который перебор по `--sweep` ответить не может: сколько стоит
# каждая часть по отдельности и упирается ли она в память или в счёт. Столбец
# «флоп/байт» против перелома 241 показывает это прямо; столбец «мс» —
# настоящее время на чипе, а не оценка.
if PARTS:
    run(sys.executable, "evals/parts.py", "--preset", "default",
        "--batch-size", "4", "--seq-len", "2048", check=False)

## 6б. Снимок из прошлой сессии

Output ноутбука переживает контейнер: всё, что лежало в `/kaggle/working` к
концу прогона, Kaggle сохраняет как output версии. Сессия 29.08.2026 оставила
там `step-33477` — 19,477 шагов, которые на Hub не уехали из-за квоты.

Чтобы их поднять, output прошлой версии надо подключить входом: **Add Input →
Your Work → Notebook Output**. Тогда он появится в `/kaggle/input`, и ячейка
ниже сама отправит самый свежий снимок на Hub. Заливка идёт с канала Kaggle,
то есть минуты, а не часы.

**Подключать вход стоит каждую сессию, а не только после аварии.** Заливка на
Hub квоту не возвращает: удаление папок и схлопывание истории оставляют блобы
висеть до сборщика мусора (замер 08.09.2026 — в HEAD один снимок и один
коммит, а Hub считает 65.08 ГБ, ровно семь заливок сессии 07.09). Со
подключённым входом ячейка сносит репозиторий целиком и заливает снимок
заново, и репозиторий каждую сессию начинает с честных ~4.4 ГБ (оценка по размеру модели).

Ячейка безопасна при каждом запуске: если на Hub уже лежит шаг не меньше, она
ничего не делает — и репозиторий тоже не трогает. Если вход не подключён —
говорит, что восстанавливать нечего, и идёт дальше.

In [ ]:
# --wipe: снести репозиторий целиком до заливки. Обычно так нельзя — сорвавшаяся
# отправка оставит его пустым, — но здесь источник лежит в output и никуда не
# денется, а повторить заливку можно с того же входа.
#
# Именно снос, а не ротация: удаление папок и схлопывание истории квоту не
# освобождают, блобы висят до сборщика мусора Hub. Сессия 07.09 встала на
# восьмой заливке, имея в HEAD один снимок и один коммит.
# Абляции Hub не трогают вовсе: ни чтений, ни заливок, ни чекпоинтов. Значит и
# возня с квотой им ни к чему — а стоит она до десяти минут из трёх часов.
if MODE != "pretrain":
    print(f"режим {MODE!r}: квоту не перекладываем "
          + ("(снимок только читаем)" if MODE == "anneal" else "(Hub не трогаем)"))
    tail = ""
else:
    code_, tail = run(sys.executable, "synth1/checkpoint.py", "recover", "pretrain",
                      "/kaggle/input", "--wipe", check=False)

# Вход не подключён (или снимок на Hub уже свежее) — сносить репозиторий было
# нечем, и квота осталась съеденной. Тогда перекладываем сами: скачать свежий
# снимок с Hub, снести репо, залить назад. Десять минут на канале Kaggle против
# восьми часов, которые иначе уйдут в переполненную квоту.
#
# Само решает, надо ли: если Hub считает занятым не сильно больше, чем лежит в
# снимках, ничего не делает. Между сносом и концом заливки единственный
# экземпляр лежит в /kaggle/working — то есть попадёт в output и переживёт
# контейнер.
if MODE == "pretrain" and ("восстанавливать нечего" in tail
                           or "восстанавливать не нужно" in tail):
    _, tail2 = run(sys.executable, "synth1/checkpoint.py", "compact", "pretrain",
                   check=False)
    tail += tail2

# Сколько Hub считает занятым сейчас. Печатается всегда: по списку снимков
# этого не видно, а упереться в квоту посреди восьмичасовой сессии дорого.
if MODE == "pretrain":
    run(sys.executable, "synth1/checkpoint.py", "usage", check=False)

# Не хватило места — дальше идти незачем: снимки этой сессии уедут в ту же
# переполненную квоту, и восемь часов уйдут в никуда, как 29.08 и 06.09.
# Лучше умереть здесь, на третьей минуте.
if "НЕ ХВАТИЛО МЕСТА" in tail:
    raise SystemExit(chr(10).join([
        "На Hub нет места, обучение не начинаем.",
        "Снимок цел: он в /kaggle/working и уедет в output этой версии.",
        "Репозиторий уже снесён, значит место съели соседние — снести их:",
        "huggingface.co/settings/storage, затем запустить сессию заново",
        "с подключённым входом-output.",
    ]))

## 7. Обучение

Дальше всё идёт само, сколько бы раз ячейку ни запускали. Что умеет
`train/session.py`, кроме как считать шаги:

* **не влезло в память** — батч вдвое меньше и заново; так до восьми;
* **сорвался лосс** — откат к последнему снимку с Hub и обход тех данных, на
  которых сорвались (позиция в корпусе лежит в снимке, поэтому без сдвига
  прогон второй раз получит тот же батч и тот же nan);
* **кончилось время** — выход с чекпоинтом, следующая сессия продолжит.

Снимок пишется на диск каждые 2000 шагов, а на Hub уезжает каждые 8000 и в
конце сессии обязательно. Разница не косметическая: диск бесплатный, а каждая
заливка стоит своих ~4.4 ГБ квоты и не возвращает их (см. 6б). Сессия 07.09
залила семь снимков, съела 65 ГБ и встала на восьмом, не досчитав полутора
часов.

Между заливками страховка — не Hub, а output: всё, что лежит в
`/kaggle/working` к концу прогона, Kaggle сохраняет вместе с версией, и
следующая сессия поднимет снимок оттуда (ячейка 6б). Терять нечего в любом
случае: в снимке лежит всё для точного продолжения — веса, momentum, позиция в
корпусе и шаг расписания.

In [ ]:
# Вся логика — в train/session.py: она в репозитории, покрыта тестами и
# обновляется вместе с кодом. Ячейка только отдаёт ей остаток времени.
# Проба LR отработала 05.09.2026, LR понижен втрое, и сессия 07.09 показала,
# чего эта проба стоит: понижение дало ступеньку вниз на -0.21 за 3,550 шагов
# и ровное плато на следующих 16,000. Проба меряет ступеньку, а не спуск, —
# третье понижение купит третью ступеньку.
#
# Поэтому включать её снова, когда лосс опять встанет, НЕ надо: она ответит
# «понижай», это будет неправдой, и полтора часа сессии уйдут впустую.
# Что мерить вместо неё — STATUS.md, «Что открыто».
PROBE = []

if MODE == "ablations":
    # Батарее отдаётся меньшее из двух: остаток сессии и бюджет квоты.
    # Десятая часа снимается на печать таблицы — она в самом конце, и потерять
    # её из-за убитой сессии значит потерять всю батарею.
    left = max(left_hours() - 0.1, 0.1)
    budget = min(left, BUDGET_HOURS) if BUDGET_HOURS else left
    print(f"батарея абляций: бюджет {budget:.2f} ч "
          f"(остаток сессии {left:.2f} ч, лимит квоты {BUDGET_HOURS or '—'})")
    run(sys.executable, "evals/ablate.py", "--hours", f"{budget:.3f}", check=False)
    # Проба на настоящей модели: батарея отвечает про 88M от нуля, а решение
    # принимается про 579M на плато. Это разные вопросы, и второй дороже.
    if PROBE_LR:
        left = max(left_hours() - 0.2, 0.0)
        if left < 0.8:
            print(f"на пробу LR не осталось времени ({left:.2f} ч) — пропуск")
        else:
            run(sys.executable, "train/session.py",
                "--hours", f"{left:.2f}", "--batch-size", "32",
                "--probe-lr", str(PROBE_LR), "--probe-only", check=False)
elif MODE == "final":
    print(f"настоящий отжиг: {FINAL_ANNEAL_STEPS:,} шагов под прогоном 'anneal', "
          f"первый старт со снимка 'pretrain'")
    print("для сравнения: короткие отжиги на 2000 шагов дали 1.9170 (84,425) "
          "и 1.9195 (113,780)")
    run(sys.executable, "train/session.py",
        "--hours", f"{left_hours():.2f}", "--batch-size", "32",
        "--run", "anneal", "--anneal", str(FINAL_ANNEAL_STEPS),
        "--init-from", "pretrain", check=False)
elif MODE == "chat":
    # Поговорить с моделью самому: веб-чат с поиском по Википедии. Ссылка
    # вида https://….gradio.live появится в выводе ниже — открыть в браузере.
    # Работает, пока идёт сессия (запускать в редакторе, не Save & Run All).
    run(sys.executable, "-m", "pip", "install", "-q", "gradio", check=False)
    run(sys.executable, "demo/chat.py", "--ui", "gradio", check=False)
elif MODE == "auto":
    import json as _json
    from huggingface_hub import hf_hub_download

    def stage_state(st):
        # (шаг, конец этапа) по последнему снимку на Hub; (None, None) — нет.
        repo = f"sekund0chka/synth-1-{st['run']}"
        try:
            names = HfApi(token=os.environ.get("HF_TOKEN")).list_repo_files(repo)
        except Exception:
            return None, None
        steps_ = sorted(int(f.split("/")[0][5:]) for f in names
                        if f.startswith("step-") and f.endswith("meta.json"))
        if not steps_:
            return None, None
        meta = _json.loads(open(hf_hub_download(
            repo, f"step-{steps_[-1]}/meta.json",
            token=os.environ.get("HF_TOKEN"))).read())
        start = meta.get("anneal_from")
        return steps_[-1], (start + st["steps"]) if start is not None else None

    done_now, verdict = [], ""
    for st in STAGES:
        if st.get("retired"):
            print(f"этап {st['run']!r}: выведен из плана", flush=True)
            continue
        step_, end_ = stage_state(st)
        if step_ is not None and end_ is not None and step_ >= end_:
            print(f"этап {st['run']!r}: закончен (шаг {step_:,})", flush=True)
            continue
        if left_hours() < 1.0:
            verdict = (f"время сессии кончилось, этап {st['run']!r} продолжит "
                       f"следующий запуск")
            break
        print(f"этап {st['run']!r}: "
              + (f"продолжаем с шага {step_:,} до {end_:,}" if step_ else
                 f"начинаем со снимка {st['init']!r}"), flush=True)
        # Локальный диск — 20 ГБ, снимок — 4.4: перед этапом оставляем только
        # его самого и родителя. 29.10 замеры накачали d2, f, g, и fv2 упал на
        # скачивании родителя восемь раз подряд.
        import pathlib as _pathlib
        import shutil as _shutil
        for old_ in (_pathlib.Path(WORK) / "checkpoints").glob("*"):
            if old_.name not in (st["run"], st["init"]):
                _shutil.rmtree(old_, ignore_errors=True)
        if st.get("prepare"):
            code_, _ = run(sys.executable, *st["prepare"], check=False)
            if code_:
                verdict = (f"подготовка данных этапа {st['run']!r} упала — "
                           f"нужен разбор лога")
                break
        batch_ = str(32 * 2048 // st["seq"])
        run(sys.executable, "train/session.py",
            "--hours", f"{left_hours() - 0.6:.2f}",
            "--batch-size", batch_, "--seq-len", str(st["seq"]),
            "--mixture", st["mixture"], "--run", st["run"],
            "--init-from", st["init"], "--new-stage",
            "--anneal", str(st["steps"]), "--lr-scale", str(st["lr"]),
            "--warmup", str(st["warmup"]), "--push-every", "12000",
            check=False)
        step_, end_ = stage_state(st)
        if step_ is None or end_ is None or step_ < end_:
            verdict = (f"этап {st['run']!r} не дошёл до конца (шаг {step_}) — "
                       f"следующий запуск продолжит; если это повторяется, "
                       f"нужен разбор лога")
            break
        done_now.append(st["run"])
        # Снимки прошлых этапов уже на Hub, а диск — 20 ГБ на всё: цепочка
        # 23.10 дошла до 1.8 ГБ свободных, и запись снимка g шла 553 с.
        # Замер, которому нужен старый снимок, скачает его заново (~20 с).
        import pathlib as _pathlib
        import shutil as _shutil
        for old_ in (_pathlib.Path(WORK) / "checkpoints").glob("*"):
            if old_.name != st["run"]:
                _shutil.rmtree(old_, ignore_errors=True)
        if st.get("eval") == "d1" and left_hours() > 0.4:
            run(sys.executable, "evals/d1.py", "--runs", st["init"], st["run"],
                "--n", "128", check=False)
        if st.get("eval") == "d2" and left_hours() > 0.4:
            # D1 после D2 — не разучилась ли отвечать по документам; D2 —
            # начинает ли с поиска.
            run(sys.executable, "evals/d1.py", "--runs", st["run"],
                "--n", "128", check=False)
            run(sys.executable, "evals/d2.py", "--runs",
                st.get("baseline", st["init"]), st["run"], "--n", "64", check=False)
            if st.get("env") and left_hours() > 0.6:
                # Быстрый замер со средой сразу после D2: виден ли сдвиг до E/F/G.
                run(sys.executable, "evals/d2_env.py", "--runs", st["run"],
                    *st["env"], check=False)
        if st.get("eval") == "e" and left_hours() > 0.4:
            # Калибровка до и после.
            run(sys.executable, "evals/e.py", "--runs",
                st.get("baseline", st["init"]), st["run"], check=False)
        if st.get("eval") == "f" and left_hours() > 0.4:
            # Отказы, выдумки и калибровка после ORPO.
            run(sys.executable, "evals/e.py", "--runs",
                *([st["baseline"]] if "baseline" in st else []), st["run"],
                check=False)
        if st.get("eval") == "g" and left_hours() > 0.4:
            # Личность; поиск в конце цепочки — не потерян ли.
            run(sys.executable, "evals/g.py", "--runs",
                st.get("baseline", st["init"]), st["run"], check=False)
            run(sys.executable, "evals/d2.py", "--runs", st["run"],
                "--n", "64", check=False)
        if st.get("eval") == "g" and left_hours() > 0.8:
            # Критерий D2 с поисковиком — один раз, в конце цепочки: начало
            # (d2) против конца (g). Самый долгий замер (data/search_env.py).
            run(sys.executable, "evals/d2_env.py", "--runs",
                *([st["env_baseline"]] if st.get("env_baseline") else []),
                st["run"], check=False)
        if st.get("eval") == "g" and left_hours() > 0.6:
            # Как видит пользователь: разговоры целиком на корпусе демо-чата.
            run(sys.executable, "evals/demo.py", "--runs", st["run"], check=False)
    else:
        verdict = ("все этапы плана пройдены (третья версия D2 → G); дальше — "
                   "разбор замеров")
    print()
    print("АВТОПИЛОТ: " + (f"сделано {', '.join(done_now)}; " if done_now else
                           "в этой сессии этапов не закончено; ") + verdict,
          flush=True)
elif MODE == "context":
    # Сначала — замер шага на той форме, на которой пойдёт этап: прежняя
    # раскладка (XLA сам), shard_map со слотами через scatter и через gather.
    # Десять минут, и этап идёт на самом быстром из путей shard_map.
    # Профиль самого быстрого ложится в output — для следующего разбора.
    CB = str(32 * 2048 // CONTEXT_SEQ)
    timing = {}
    for label, extra in (("XLA сам, 2048", ["--batch-size", "32", "--seq-len", "2048",
                                            "--auto-sharding", "--steps", "5"]),
                         ("shard_map, scatter, 2048", ["--batch-size", "32", "--seq-len", "2048"]),
                         ("shard_map, gather, 2048", ["--batch-size", "32", "--seq-len", "2048",
                                                      "--set", "moe_dispatch=gather"]),
                         ("shard_map, scatter", ["--batch-size", CB, "--seq-len", str(CONTEXT_SEQ)]),
                         ("shard_map, gather", ["--batch-size", CB, "--seq-len", str(CONTEXT_SEQ),
                                                "--set", "moe_dispatch=gather",
                                                "--out", "/kaggle/working/prof_context"])):
        _, out = run(sys.executable, "evals/profile.py", *extra, check=False)
        got = re.search(r"шаг ([\d.]+) мс", out)
        timing[label] = float(got.group(1)) if got else None
    print("замер шага:", timing, flush=True)
    fast_gather = (timing.get("shard_map, gather") or 1e9) < (timing.get("shard_map, scatter") or 1e9)
    DISPATCH = "gather" if fast_gather else "scatter"
    print(f"слоты экспертов: {DISPATCH}", flush=True)
    print(f"растяжка контекста: {CONTEXT_STEPS:,} шагов на длине {CONTEXT_SEQ}, "
          f"прогон 'context', первый старт со снимка 'anneal'")
    run(sys.executable, "train/session.py", "--set", f"moe_dispatch={DISPATCH}",
        "--hours", f"{left_hours() - 0.35:.2f}",
        "--batch-size", str(32 * 2048 // CONTEXT_SEQ),
        "--seq-len", str(CONTEXT_SEQ), "--mixture", "anneal",
        "--run", "context", "--init-from", "anneal", "--new-stage",
        "--anneal", str(CONTEXT_STEPS), "--lr-scale", str(CONTEXT_LR),
        "--warmup", "500", "--push-every", "12000", check=False)
    # Отложенный лосс на той же линейке, что у плато и отжига (2048, смесь
    # pretrain): эталон отжига 1.8300. И примеры продолжений обоих снимков.
    run(sys.executable, "evals/report.py", "--runs", "anneal", "context",
        check=False)
elif MODE == "anneal":
    # Отжиг с того места, где стоит снимок. Снимки не пишутся и на Hub ничего
    # не уезжает: нам нужно число, а не ветка обучения.
    left = max(left_hours() - 0.2, 0.1)
    print(f"отжиг {ANNEAL_STEPS} шагов на настоящей модели, остаток {left:.2f} ч")
    run(sys.executable, "train/pretrain.py",
        "--run", "pretrain", "--preset", "default", "--mixture", "pretrain",
        "--seq-len", "2048", "--batch-size", "32",
        "--steps", str(ANNEAL_STEPS), "--anneal", str(ANNEAL_STEPS),
        "--hours", f"{left:.2f}", "--log-every", "50", "--eval-every", "100",
        "--show-every", "0", "--resume", "--no-checkpoint", check=False)
else:
    if ANNEAL_FIRST:
        # Тот же вызов, что в режиме "anneal", но с потолком по времени: отжиг
        # не должен съесть сессию, если что-то пойдёт не так. Точка
        # продолжения не сдвигается (--no-checkpoint), и претрейн ниже
        # стартует с того же снимка.
        left = min(max(left_hours() - 0.5, 0.1), 1.2)
        print(f"сперва отжиг {ANNEAL_STEPS} шагов с текущего снимка "
              f"(потолок {left:.2f} ч), затем претрейн с того же снимка")
        print("эталон: снимок 84,425 → 1.9170. Ниже эталона — плато копило; "
              "около или выше — эти шаги ничего не дали")
        run(sys.executable, "train/pretrain.py",
            "--run", "pretrain", "--preset", "default", "--mixture", "pretrain",
            "--seq-len", "2048", "--batch-size", "32",
            "--steps", str(ANNEAL_STEPS), "--anneal", str(ANNEAL_STEPS),
            "--hours", f"{left:.2f}", "--log-every", "50", "--eval-every", "100",
            "--show-every", "0", "--resume", "--no-checkpoint", check=False)
        print()
        print("отжиг закончен, дальше обычный претрейн")
    run(sys.executable, "train/session.py",
        "--hours", f"{left_hours():.2f}", "--batch-size", "32", *PROBE, check=False)

print()
print(f"сессия заняла {(time.time() - T0) / 3600:.2f} ч")

## Если что-то пошло не так

Ячейки ниже руками не нужны — они для разбора, когда лог покажет странное.

`Device or resource busy` означает, что чип держит другой процесс: упавший
прогон или само ядро, если в нём импортировали jax. Если виновато ядро — его
надо перезапустить (**Run → Restart session**).

Кончившийся диск выглядит не как ошибка обучения: сначала `OSError: [Errno 28]`
из papermill, который не смог дописать ноутбук, потом `UnicodeDecodeError` на
обрубке. Историю снимков это не рушит — она на Hub, локальная папка лишь буфер.

In [ ]:
print(sh("bash", "-lc", "ls -l /dev/vfio/ 2>/dev/null; fuser -v /dev/vfio/0 2>&1 | head -20"))
print(sh("bash", "-lc", "df -h /kaggle/working; du -sh /kaggle/working/synth-1/checkpoints/* 2>/dev/null"))
# Снять зависшие прогоны и локальные снимки (на Hub они уже есть):
# print(sh("bash", "-lc", "pkill -f train/pretrain.py; rm -rf /kaggle/working/synth-1/checkpoints/*"))

## Ручные прогоны

Ничего из этого при Save & Run All не выполняется — ячеек здесь нет намеренно.
Команды оставлены как справка, запускать их можно в интерактивной сессии:

    # железо считает? случайные токены, снимков не пишет
    python train/pretrain.py --run smoke --preset default --fake-data \
        --steps 30 --seq-len 2048 --batch-size 8 --log-every 5 --no-checkpoint

    # абляции: сорок минут, модель поменьше, один корпус и seed
    python train/pretrain.py --run abl-dense --preset ablation-dense \
        --mixture pretrain --seq-len 2048 --batch-size 16 --steps 2000 \
        --total-steps 2000 --log-every 50 --checkpoint-every 1000 --show-every 500

    python train/pretrain.py --run abl-moe --preset ablation \
        --mixture pretrain --seq-len 2048 --batch-size 16 --steps 2000 \
        --total-steps 2000 --log-every 50 --checkpoint-every 1000 --show-every 500

    # замеры на отложенных шардах
    python evals/bench.py --run pretrain

    # высоко ли плато LR: два прогона по тысяче шагов с одного снимка.
    # Смотреть надо на строку «отложенный лосс» — обучающий шумит сильнее,
    # чем меняется.
    python train/pretrain.py --run pretrain --resume --no-push \
        --steps 1000 --batch-size 32 --eval-every 250 --lr-scale 1.0
    python train/pretrain.py --run pretrain --resume --no-push \
        --steps 1000 --batch-size 32 --eval-every 250 --lr-scale 0.3

    # разбор завалов на Hub (снимок ~4.4 ГБ)
    python synth1/checkpoint.py list  pretrain
    python synth1/checkpoint.py prune pretrain 1

    # поднять снимок из output прошлой сессии (вход подключается через Add Input)
    python synth1/checkpoint.py recover pretrain /kaggle/input --wipe

    # что Hub считает занятым: в квоту идёт история, а не HEAD
    python synth1/checkpoint.py usage